In [2]:
import pandas as pd   # for handling tabular data
import numpy as np    # for fast numerical operations

In [3]:
ge1 = pd.read_excel("../data/raw/GSE33000_raw_data_2 class_PFC_dataset_ClassDenoted (2).xlsx")

# Load the gene expression dataset into a pandas DataFrame
# Excel is used because data is stored in .xlsx format

In [4]:
ge1 = ge1.iloc[:, :-1]

# Last column contains an invalid value (157) only in label row
# It is not a real sample → must be removed

In [6]:
ge1_labels = ge1.iloc[-1, 1:].values.astype(int)
ge1_data = ge1.iloc[:-1].copy()

# Last row = class labels (0/1)
# Remaining rows = gene expression data
# .copy() avoids pandas warnings

In [9]:
# Check last 5 column names
print(ge1.columns[-5:])

# Check last row (labels) for last 5 columns
print(ge1.iloc[-1, -5:])

# Check unique values in last row
print(pd.unique(ge1.iloc[-1]))

Index(['PFC_463', 'PFC_464', 'PFC_465', 'PFC_466', 'PFC_467'], dtype='object')
PFC_463    0.0
PFC_464    0.0
PFC_465    0.0
PFC_466    0.0
PFC_467    0.0
Name: 39280, dtype: object
['Class Lebel' np.float64(1.0) np.float64(0.0)]


In [10]:
ge1_data['Gene'] = ge1_data['Gene'].fillna('Unknown')

# WHY:
# Some gene names are missing (NaN)
# Replace with 'Unknown' so indexing does not break later

In [11]:
print(ge1_data['Gene'].isnull().sum())

0


In [12]:
ge1_data.set_index('Gene', inplace=True)

# Gene names should act as feature identifiers (column labels after transpose)
# This removes the text column and keeps only numeric data

In [13]:
print(ge1_data.index[:5])   # check first few gene names
print(ge1_data.shape)      # check shape

Index(['RSE_00000118999', 'AGRN,FLJ45064', 'HMGB2,HMG2', 'LOC285706',
       'DNAH12,DHC3,HL19,DLP12,HL-19,hdhc3,DNAHC3,DNAHC12'],
      dtype='object', name='Gene')
(39280, 467)


In [14]:
ge1_data = pd.DataFrame(ge1_data.values.T, columns=ge1_data.index)

# Current: Genes × Samples  → (39280, 467)
# Needed:  Samples × Genes  → (467, 39280)
# ML models require rows = samples, columns = features
# Using NumPy transpose is faster than pandas .T for large data

In [15]:
print(ge1_data.shape)

(467, 39280)


In [16]:
ge1_data = ge1_data.astype(np.float32)

# Ensure all values are numeric
# float32 reduces memory usage (important for large dataset)
# Prevents computation issues later

In [17]:
print(ge1_data.dtypes.unique())

[dtype('float32')]


In [18]:
data_np = ge1_data.values

# NumPy arrays are much faster than pandas for large computations
# Needed for efficient mean imputation

In [19]:
print(type(data_np))
print(data_np.shape)

<class 'numpy.ndarray'>
(467, 39280)


In [20]:
col_mean = np.nanmean(data_np, axis=0)

# Compute mean for each gene (column)
# Required to replace missing values (mean imputation)
# np.nanmean ignores NaN values automatically

In [21]:
print(col_mean.shape)
print(np.isnan(col_mean).sum())

(39280,)
0


In [22]:
inds = np.where(np.isnan(data_np))

# Find positions (row, column) where values are missing (NaN)
# We will replace these using column mean

In [23]:
print(len(inds[0]))

250991


In [24]:
data_np[inds] = np.take(col_mean, inds[1])

# inds[0] → row indices (samples)
# inds[1] → column indices (genes)
# np.take(col_mean, inds[1]) → picks correct mean for each gene
# This replaces each NaN with its column (gene) mean

In [25]:
print(np.isnan(data_np).sum())

0


In [26]:
ge1_data = pd.DataFrame(data_np, columns=ge1_data.columns)

# WHY:
# We used NumPy for fast computation
# Now convert back to pandas for easier handling
# Keep original gene names as column names

In [27]:
print(type(ge1_data))
print(ge1_data.shape)

<class 'pandas.core.frame.DataFrame'>
(467, 39280)


In [29]:
ge1_data.columns = ge1_data.columns.astype(str)

# Ensure all feature names are strings
# Required for sklearn compatibility

In [30]:
print(type(ge1_data.columns[0]))

<class 'str'>


In [31]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
ge1_data = scaler.fit_transform(ge1_data)

# Apply Z-score normalization
# Mean = 0, Standard deviation = 1
# Ensures all genes are on same scale
# Improves ML model performance

In [32]:
print(type(ge1_data))
print(ge1_data.shape)
print(round(ge1_data.mean(), 2))
print(round(ge1_data.std(), 2))

<class 'numpy.ndarray'>
(467, 39280)
0.0
1.0


In [33]:
np.save("../data/processed/GSE33000_data.npy", ge1_data)
np.save("../data/processed/GSE33000_labels.npy", ge1_labels)

# Clear naming → easy to identify dataset later
# Important when working with multiple datasets (GE + DM)

In [34]:
import os
print(os.listdir("../data/processed"))

['.gitkeep', 'GSE33000_data.npy', 'GSE33000_labels.npy']
